<a href="https://colab.research.google.com/github/phenomenaldatalab/Foundations-of-AI-for-Business/blob/main/Exercise9_Aggregation%20II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# 7x7 Grid definition
GRID_SIZE = 9

# Define feature regions (row, col coordinates)
# row 0 is top, row 6 is bottom. col 0 is left, col 6 is right.
FEATURES = {
    "Vert-Left":   [(r, 0) for r in range(GRID_SIZE)],
    "Vert-Center": [(r, 3) for r in range(GRID_SIZE)],
    "Vert-Right":  [(r, 6) for r in range(GRID_SIZE)],
    "Horiz-Top":   [(0, c) for c in range(GRID_SIZE)],
    "Horiz-Mid":   [(3, c) for c in range(GRID_SIZE)],
    "Horiz-Bot":   [(6, c) for c in range(GRID_SIZE)],
}

# Cognitive weights (expected = 1.0, unexpected = penalize)
LETTERS = {
    "E": {"expected": ["Vert-Left", "Horiz-Top", "Horiz-Mid", "Horiz-Bot"], "unexpected": ["Vert-Center", "Vert-Right"]},
    "F": {"expected": ["Vert-Left", "Horiz-Top", "Horiz-Mid"], "unexpected": ["Horiz-Bot", "Vert-Center", "Vert-Right"]},
    "L": {"expected": ["Vert-Left", "Horiz-Bot"], "unexpected": ["Horiz-Top", "Horiz-Mid", "Vert-Center", "Vert-Right"]},
    "H": {"expected": ["Vert-Left", "Vert-Right", "Horiz-Mid"], "unexpected": ["Horiz-Top", "Horiz-Bot", "Vert-Center"]},
    "T": {"expected": ["Horiz-Top", "Vert-Center"], "unexpected": ["Vert-Left", "Vert-Right", "Horiz-Mid", "Horiz-Bot"]},
}

class PandemoniumApp:
    def __init__(self):
        self.grid_size = GRID_SIZE
        self.buttons = []
        self.out = widgets.Output()

        # Grid state (0 = empty, 1 = colored)
        self.grid_state = np.zeros((self.grid_size, self.grid_size))

        # Create grid UI using ipywidgets ToggleButtons
        grid_rows = []
        for r in range(self.grid_size):
            row_buttons = []
            for c in range(self.grid_size):
                btn = widgets.ToggleButton(
                    value=False,
                    layout=widgets.Layout(width='32px', height='32px', margin='1px'),
                    style={'button_color': '#f0f0f0'}
                )
                btn.observe(self.make_callback(r, c), names='value')
                row_buttons.append(btn)
                self.buttons.append(btn)
            grid_rows.append(widgets.HBox(row_buttons))

        self.grid_ui = widgets.VBox(grid_rows)

        # Control elements
        self.clear_btn = widgets.Button(description="Clear Grid", button_style='warning', layout=widgets.Layout(width='120px'))
        self.clear_btn.on_click(self.clear_grid)

        self.noise_slider = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description="Noise Level:", layout=widgets.Layout(width='250px'))
        self.noise_btn = widgets.Button(description="Add Noise", button_style='info', layout=widgets.Layout(width='100px'))
        self.noise_btn.on_click(self.add_noise)

        self.control_ui = widgets.HBox([self.clear_btn, self.noise_slider, self.noise_btn], layout=widgets.Layout(margin='10px 0px 0px 0px'))

        # Assemble UI Layout
        self.ui = widgets.VBox([
            widgets.HTML("<h2>👿 Pandemonium Model Simulator (Colab) 👿</h2>"),
            widgets.HTML("<p><i>Click the grid buttons below to draw letters (E, F, L, H, T) and see how the demons respond!</i></p>"),
            widgets.HBox([self.grid_ui, self.out]),
            self.control_ui
        ])

        # Run recognition on setup
        with self.out:
            self.update_recognition()

    def make_callback(self, r, c):
        def callback(change):
            self.grid_state[r, c] = 1.0 if change['new'] else 0.0
            # Visually highlight active buttons in the grid
            btn_idx = r * self.grid_size + c
            self.buttons[btn_idx].style.button_color = '#3f51b5' if change['new'] else '#f0f0f0'
            self.update_recognition()
        return callback

    def clear_grid(self, b):
        self.grid_state.fill(0)
        for btn in self.buttons:
            # Temporarily disable observers to avoid triggering update on every clear
            btn.unobserve_all()
            btn.value = False
            btn.style.button_color = '#f0f0f0'

        # Re-attach observers
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                idx = r * self.grid_size + c
                self.buttons[idx].observe(self.make_callback(r, c), names='value')

        self.update_recognition()

    def add_noise(self, b):
        noise_level = self.noise_slider.value
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                if np.random.rand() < noise_level:
                    idx = r * self.grid_size + c
                    # This will toggle the value, triggering the observer callback
                    self.buttons[idx].value = not self.buttons[idx].value
        self.update_recognition()

    def update_recognition(self):
        with self.out:
            clear_output(wait=True)

            # 1. Feature Demons Activation
            features_act = {}
            for name, coords in FEATURES.items():
                active = sum(self.grid_state[r, c] for r, c in coords)
                features_act[name] = (active / len(coords)) * 100

            # 2. Cognitive Demons Activation (Shouting Volume)
            cognitive_act = {}
            for letter, config in LETTERS.items():
                pos_sum = sum(features_act[f] for f in config["expected"])
                neg_sum = sum(features_act[f] for f in config["unexpected"])

                score = (pos_sum / len(config["expected"])) - (neg_sum / len(config["unexpected"])) * 0.5
                cognitive_act[letter] = max(0.0, min(100.0, score))

            # 3. Decision Demon Selects the Loudest
            winner = max(cognitive_act, key=cognitive_act.get)
            decision = winner if cognitive_act[winner] > 15 else "None"

            # Plot the activations in real time
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

            # Plot Feature Demons
            y_pos1 = np.arange(len(features_act))
            ax1.barh(y_pos1, list(features_act.values()), color='#2196F3')
            ax1.set_yticks(y_pos1)
            ax1.set_yticklabels(list(features_act.keys()))
            ax1.set_xlim(0, 100)
            ax1.set_title("Feature Demons (Shouting)")
            ax1.set_xlabel("Activation (%)")

            # Plot Cognitive Demons
            y_pos2 = np.arange(len(cognitive_act))
            colors = ['#FF5722' if k == decision else '#9E9E9E' for k in cognitive_act.keys()]
            ax2.barh(y_pos2, list(cognitive_act.values()), color=colors)
            ax2.set_yticks(y_pos2)
            ax2.set_yticklabels(list(cognitive_act.keys()))
            ax2.set_xlim(0, 100)
            ax2.set_title("Cognitive Demons (Shouting)")
            ax2.set_xlabel("Activation (%)")

            plt.tight_layout()
            plt.show()

            # Text output representing Decision Demon
            print(f"🔊 Decision Demon says: \"I hear the loudest shouting from {decision}! ({cognitive_act[winner]:.1f}% volume)\"")

# Launch function
def launch():
    app = PandemoniumApp()
    display(app.ui)

if __name__ == '__main__':
    launch()
